In [1]:
import pysindy as ps

https://pysindy.readthedocs.io/en/latest/generated/pysindy.optimizers.STLSQ.html
Statistiek, zie : Regressie.pdf Statistiek 1, KUL 2425 semester 2
en ook
https://apxml.com/courses/probability-statistics-essentials-ml/chapter-6-introduction-regression-analysis/regression-evaluation-metrics

In [ ]:
import numpy as np
from scipy.integrate import odeint
from pysindy import SINDy
from pysindy.optimizers import STLSQ
lorenz = lambda z,t : [10*(z[1] - z[0]),
                       z[0]*(28 - z[2]) - z[1],
                       z[0]*z[1] - 8/3*z[2]]
t = np.arange(0,2,.002)
x = odeint(lorenz, [-8,8,27], t)
opt = STLSQ(threshold=.1, alpha=.5)
model = SINDy(optimizer=opt)
model.fit(x, t=t[1]-t[0])
model.print()

Note: the base optimizer for SINDy is STLSQ, of which the default parameters are set in _init_ of the STLSQ class in PySINDy

Any instance of the SINDy class initializes with:

    def __init__(
        self,
        optimizer: Optional[BaseOptimizer] = None,
        feature_library: Optional[BaseFeatureLibrary] = None,
        differentiation_method: Optional[BaseDifferentiation] = None,
    ):
        if optimizer is None:
            optimizer = STLSQ()
        self.optimizer = optimizer
        if feature_library is None:
            feature_library = PolynomialLibrary()
        self.feature_library = feature_library
        if differentiation_method is None:
            differentiation_method = FiniteDifference(axis=-2)
        self.differentiation_method = differentiation_method


Instances of the SINDy class have an attribute "optimizer" and the method "fit"
- This method contains (line 408) : self.optimizer.fit(x, x_dot)
By default optimizer = STLSQ, so it is reasonable to dig into the code of STLSQ

STLSQ is implemented as a class whose instances initialize with:


    def __init__(
        self,
        threshold=0.1,
        alpha=0.05,
        max_iter=20,
        ridge_kw=None,
        normalize_columns=False,
        copy_X=True,
        initial_guess=None,
        verbose=False,
        sparse_ind=None,
        unbias=True,
    ):
        super().__init__(
            max_iter=max_iter,
            copy_X=copy_X,
            normalize_columns=normalize_columns,
            unbias=unbias,
        )

        if threshold < 0:
            raise ValueError("threshold cannot be negative")
        if alpha < 0:
            raise ValueError("alpha cannot be negative")

        self.threshold = threshold
        self.alpha = alpha
        self.ridge_kw = ridge_kw
        self.initial_guess = initial_guess
        self.verbose = verbose
        self.sparse_ind = sparse_ind


method of STLSQ for sparsifying... and regression method

    def _sparse_coefficients(
        self, dim: int, ind_nonzero: np.ndarray, coef: np.ndarray, threshold: float
    ) -> (np.ndarray, np.ndarray):
        """Perform thresholding of the weight vector(s) (on specific indices
        if ``self.sparse_ind`` is not None)"""
        c = np.zeros(dim)
        c[ind_nonzero] = coef
        big_ind = np.abs(c) >= threshold
        if self.sparse_ind is not None:
            nonsparse_ind_mask = np.ones_like(ind_nonzero)
            nonsparse_ind_mask[self.sparse_ind] = False
            big_ind = big_ind | nonsparse_ind_mask
        c[~big_ind] = 0
        return c, big_ind


    def _regress(self, x: np.ndarray, y: np.ndarray, dim: int, sparse_sub: np.ndarray):
        """Perform the ridge regression (on specific indices if
        ``self.sparse_ind`` is not None)"""
        kw = self.ridge_kw or {}
        if self.sparse_ind is None:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=LinAlgWarning)
                try:
                    coef = ridge_regression(x, y, self.alpha, **kw)
                except LinAlgWarning:
                    # increase alpha until warning stops
                    self.alpha = 2 * self.alpha
            self.iters += 1
            return coef
        if self.sparse_ind is not None:
            alpha_array = np.zeros((dim, dim))
            alpha_array[sparse_sub, sparse_sub] = np.sqrt(self.alpha)
            x_lin = np.concatenate((x, alpha_array), axis=0)
            y_lin = np.concatenate((y, np.zeros((dim,))))
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=LinAlgWarning)
                try:
                    coef = (
                        LinearRegression(fit_intercept=False, **kw)
                        .fit(x_lin, y_lin)
                        .coef_
                    )
                except LinAlgWarning:
                    # increase alpha until warning stops
                    self.alpha = 2 * self.alpha
            self.iters += 1
            return coef
